# 1. Preparação Geral dos Dados e Constantes

Nesta etapa, carregamos a base limpa do NHIS 2025 (`adult25_limpo.csv`), definimos o `ID_DOMICILIO` como índice e isolamos a variável alvo multiclasse (`SEVERIDADE_DEPRESSAO_PHQ`), filtrando os registros indefinidos na Target (valores `-1` e `5`).

Além disso, isolamos a coluna `PESO_AMOSTRAL` e removemos do conjunto de preditores ($X$) todas as colunas de sintomas diretos de depressão para evitar vazamento de dados (*data leakage*).

In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==============================================================================
# CONFIGURAÇÃO DE VARIÁVEIS GLOBAIS / CONSTANTES
# ==============================================================================
ID_COLUNA = 'ID_DOMICILIO'
PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET = 'SEVERIDADE_DEPRESSAO_PHQ'

COLUNAS_SINTOMAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM',
    'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM',
    'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM',
    'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM',
    'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO',
    'REMEDIO_DEPRESSAO',
    'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO'
]

# 1. Carregar a base limpa e definir o ID como índice
df = pd.read_csv('adult25_limpo.csv').set_index(ID_COLUNA)

# 2. Eliminar os registros indefinidos no TARGET (-1 e 5)
df = df[~df[TARGET].isin([-1, 5])].copy()

# 3. Isolar Target (y), Peso Amostral e Features Preditoras (X)
y = df[TARGET]
sample_weights = df[PESO_COLUNA]
colunas_remover = [TARGET, PESO_COLUNA] + COLUNAS_SINTOMAS_DEPRESSAO
X = df.drop(columns=[col for col in colunas_remover if col in df.columns])

# 2. Cenário A: Treinamento Mantendo Códigos Indeterminados (-1) nas Features

Neste primeiro cenário, mantemos as categorias de não-resposta (`-1`) presentes nas variáveis preditoras ($X$), permitindo que o modelo de árvore e a codificação One-Hot tratem o valor como uma categoria válida.

In [21]:
# Split Estratificado
X_train_A, X_test_A, y_train_A, y_test_A, w_train_A, w_test_A = train_test_split(
    X, y, sample_weights, test_size=0.2, random_state=42, stratify=y
)

# Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [col for col in X_train_A.columns if col not in num_cols]

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_A = preprocessor_A.fit_transform(X_train_A)
X_test_proc_A = preprocessor_A.transform(X_test_A)

# Modelo Random Forest
model_A = RandomForestClassifier(random_state=42, class_weight='balanced')
model_A.fit(X_train_proc_A, y_train_A, sample_weight=w_train_A)

# Avaliação Cenário A
y_pred_A = model_A.predict(X_test_proc_A)
y_proba_A = model_A.predict_proba(X_test_proc_A)

print("--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---")
print(classification_report(y_test_A, y_pred_A, sample_weight=w_test_A, zero_division=0))

print("--- CENÁRIO A: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_A, y_pred_A, sample_weight=w_test_A)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_A, y_proba_A, multi_class='ovr', sample_weight=w_test_A):.4f}")

--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.94      0.89      0.91 40224045.76199999
           2       0.44      0.61      0.51 7209618.388000002
           3       0.44      0.25      0.32 2466480.354
           4       0.63      0.64      0.63 1334503.7569999998

    accuracy                           0.82 51234648.26099999
   macro avg       0.61      0.60      0.59 51234648.26099999
weighted avg       0.83      0.82      0.82 51234648.26099999

--- CENÁRIO A: Matriz de Confusão 4x4 ---
[[35900034  4173661   128854    21497]
 [ 2268874  4405756   425006   109983]
 [  142188  1321019   627418   375855]
 [   29742   198955   256571   849235]]
ROC-AUC Score (OvR): 0.9140


# 3. Cenário B: Treinamento Apenas com Informações Completas (Remoção de -1 no X)

Conforme discutido com a equipe e orientação do prof. Roy, realizamos um teste estrito filtrando qualquer observação que contenha não-resposta (`-1`) em qualquer feature preditora. A amostragem retida foi de **22.084 registros (93.5% dos dados)**.

In [22]:
# Filtragem Estrita das Features
mascara_sem_menos1 = (X != -1).all(axis=1)
X_strict = X[mascara_sem_menos1].copy()
y_strict = y[mascara_sem_menos1].copy()
w_strict = sample_weights[mascara_sem_menos1].copy()

print(f"Total de registros na base completa estrita: {len(X_strict)}")

# Split Estratificado Cenário B
X_train_B, X_test_B, y_train_B, y_test_B, w_train_B, w_test_B = train_test_split(
    X_strict, y_strict, w_strict, test_size=0.2, random_state=42, stratify=y_strict
)

# Pré-processamento
preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_B = preprocessor_B.fit_transform(X_train_B)
X_test_proc_B = preprocessor_B.transform(X_test_B)

# Modelo Random Forest Cenário B
model_B = RandomForestClassifier(random_state=42, class_weight='balanced')
model_B.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

# Avaliação Cenário B
y_pred_B = model_B.predict(X_test_proc_B)
y_proba_B = model_B.predict_proba(X_test_proc_B)

print("--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---")
print(classification_report(y_test_B, y_pred_B, sample_weight=w_test_B, zero_division=0))

print("--- CENÁRIO B: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_B, y_pred_B, sample_weight=w_test_B)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_B, y_proba_B, multi_class='ovr', sample_weight=w_test_B):.4f}")

Total de registros na base completa estrita: 22084
--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.94      0.89      0.91 37773204.54699994
           2       0.42      0.60      0.49 6824341.566000004
           3       0.28      0.19      0.23 2187297.4779999997
           4       0.62      0.52      0.57 1212457.585

    accuracy                           0.81 47997301.17599995
   macro avg       0.56      0.55      0.55 47997301.17599995
weighted avg       0.83      0.81      0.81 47997301.17599995

--- CENÁRIO B: Matriz de Confusão 4x4 ---
[[33571356  3965806   230880     5162]
 [ 2156776  4069561   515537    82468]
 [  116638  1355470   420541   294648]
 [       0   233664   349171   629622]]
ROC-AUC Score (OvR): 0.9055
